<a href="https://colab.research.google.com/github/nexageapps/AI/blob/main/Basic/B12%20-%20Byte%20Pair%20Encoding%20(BPE).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# Lesson 4: Byte Pair Encoding (BPE) - Tokenization for NLP
# ============================================================================
# Author: Karthik Arjun
# LinkedIn: https://www.linkedin.com/in/karthik-arjun-a5b4a258/
# Created: 2026
# Updated: 2026
# Reference: https://github.com/openai/tiktoken
#
# LEARNING OBJECTIVES:
# - Understand why tokenization is crucial for NLP
# - Learn the BPE algorithm step-by-step
# - Compare character-level vs subword tokenization
# - Grasp the trade-offs in vocabulary design
#
# REAL-WORLD APPLICATIONS:
# - GPT models (GPT-2, GPT-3, GPT-4)
# - Machine translation systems
# - Text generation models
# - Any modern NLP system
#
# WHY BPE?
# - Character-level: Flexible but creates LONG sequences
# - Word-level: Compact but LIMITED vocabulary (can't handle new words)
# - BPE (Subword): BEST OF BOTH WORLDS!
# ============================================================================

from collections import Counter
import re

print("="*80)
print("BYTE PAIR ENCODING (BPE) - The Tokenization Algorithm Behind GPT")
print("="*80)
print("\nBPE is a data compression technique adapted for NLP tokenization.")
print("It learns a vocabulary of subword units by iteratively merging")
print("the most frequent pairs of characters/subwords.\n")

# ============================================================================
# STEP 1: Helper Functions
# ============================================================================

def get_stats(vocab):
    """
    Count frequency of adjacent character/subword pairs.
    
    Args:
        vocab: Dictionary mapping word representations to frequencies
    
    Returns:
        Counter object with pair frequencies
    
    Example:
        Input: {'l o w </w>': 1, 'l o w e r </w>': 1}
        Output: {('l', 'o'): 2, ('o', 'w'): 2, ...}
    """
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i + 1]] += freq
    return pairs

def merge_vocab(pair, vocab):
    """
    Merge the most frequent pair in vocabulary.
    
    Args:
        pair: Tuple of (symbol1, symbol2) to merge
        vocab: Current vocabulary
    
    Returns:
        Updated vocabulary with merged pair
    
    Example:
        pair = ('l', 'o')
        'l o w' → 'lo w'
    """
    new_vocab = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)

    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab

# ============================================================================
# STEP 2: Initialize Vocabulary
# ============================================================================
# Start with character-level representation
# </w> marks end-of-word (important for distinguishing word boundaries)

text = "low lower lowest"
print("\nOriginal Text:")
print("="*80)
print(f"'{text}'\n")

# Initialize vocabulary with character-level tokens
vocab = {}
for word in text.split():
    # Add space between characters and end-of-word marker
    word_tokens = ' '.join(list(word)) + ' </w>'
    vocab[word_tokens] = vocab.get(word_tokens, 0) + 1

print("Initial Vocabulary (Character-Level):")
print("="*80)
for word, freq in vocab.items():
    print(f"  '{word}': {freq}")
print("\nNote: Each character is a separate token")
print("      </w> marks the end of a word\n")

# ============================================================================
# STEP 3: Perform BPE Merges
# ============================================================================
# Iteratively merge the most frequent pair
# This builds up a vocabulary of subword units

num_merges = 5
print(f"Performing {num_merges} BPE Merges:")
print("="*80)
print("\nAt each step, we merge the most frequent adjacent pair.")
print("This gradually builds larger subword units.\n")

for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break

    # Find most frequent pair
    best_pair = max(pairs, key=pairs.get)
    print(f"Merge {i+1}:")
    print(f"  Most frequent pair: ('{best_pair[0]}', '{best_pair[1]}') → '{best_pair[0]}{best_pair[1]}'")
    print(f"  Frequency: {pairs[best_pair]}")

    # Merge the pair
    vocab = merge_vocab(best_pair, vocab)
    
    print(f"  Updated vocabulary:")
    for word, freq in vocab.items():
        print(f"    '{word}': {freq}")
    print()

print("\nFinal Vocabulary After BPE:")
print("="*80)
for word, freq in vocab.items():
    print(f"  '{word}': {freq}")

# ============================================================================
# STEP 4: Tokenization Comparison
# ============================================================================
print("\n" + "="*80)
print("TOKENIZATION COMPARISON")
print("="*80)

test_word = "lowest"
print(f"\nOriginal word: '{test_word}'\n")

# Character-level
char_tokens = list(test_word)
print(f"1. CHARACTER-LEVEL TOKENIZATION:")
print(f"   Tokens: {char_tokens}")
print(f"   Count: {len(char_tokens)} tokens")
print(f"   Pros: Can represent ANY word")
print(f"   Cons: Very long sequences, loses word structure\n")

# Word-level
print(f"2. WORD-LEVEL TOKENIZATION:")
print(f"   Tokens: ['{test_word}']")
print(f"   Count: 1 token")
print(f"   Pros: Compact, preserves word meaning")
print(f"   Cons: Huge vocabulary, can't handle unknown words\n")

# BPE (from our final vocabulary)
for word_repr, freq in vocab.items():
    if 'l' in word_repr and 'o' in word_repr and 'w' in word_repr and 'e' in word_repr and 's' in word_repr and 't' in word_repr:
        bpe_tokens = word_repr.replace('</w>', '').split()
        print(f"3. BPE (SUBWORD) TOKENIZATION:")
        print(f"   Tokens: {bpe_tokens}")
        print(f"   Count: {len(bpe_tokens)} tokens")
        print(f"   Representation: '{word_repr}'")
        print(f"   Pros: Balanced vocabulary size, handles unknown words")
        print(f"   Cons: Slightly more complex than word-level\n")
        break

# ============================================================================
# KEY INSIGHTS FOR MASTER'S STUDENTS:
# ============================================================================
print("="*80)
print("KEY INSIGHTS")
print("="*80)
print("""
1. VOCABULARY SIZE TRADE-OFF:
   - Character-level: ~100 tokens (small vocab, long sequences)
   - Word-level: ~100,000+ tokens (huge vocab, short sequences)
   - BPE: ~30,000-50,000 tokens (balanced)

2. HANDLING UNKNOWN WORDS:
   - Word-level: Fails on new words (OOV problem)
   - BPE: Can represent ANY word using subword units
   - Example: "unbelievable" → ["un", "believ", "able"]

3. COMPRESSION VS EXPRESSIVENESS:
   - More merges → larger subwords → shorter sequences
   - Fewer merges → smaller subwords → more flexibility
   - Optimal number depends on language and task

4. MORPHOLOGICAL AWARENESS:
   - BPE naturally learns morphemes (word parts with meaning)
   - Example: "play", "playing", "played" share "play" subword
   - Helps model understand word relationships

5. MULTILINGUAL BENEFITS:
   - Works across languages with different scripts
   - Shares subwords between related languages
   - Efficient for low-resource languages

6. MODERN VARIANTS:
   - WordPiece (BERT): Similar to BPE, used by Google
   - SentencePiece: Language-independent, used by many models
   - Unigram LM: Probabilistic approach

7. PRACTICAL CONSIDERATIONS:
   - Vocabulary size affects model size and speed
   - Pre-tokenization (splitting on whitespace) matters
   - Special tokens (</w>, <unk>, <pad>) need careful handling
""")

print("="*80)
print("WHY GPT USES BPE")
print("="*80)
print("""
GPT models use BPE because it:
1. Handles rare words and typos gracefully
2. Keeps vocabulary size manageable (~50k tokens)
3. Works well across multiple languages
4. Balances sequence length and vocabulary size
5. Learns meaningful subword units automatically

This makes GPT robust and efficient for diverse text generation tasks!
""")

print("\n" + "="*80)
print("Lesson 4 Complete!")
print("="*80)
print("Next: L4A - BPE with tiktoken (OpenAI's implementation)")
print("We'll use the same tokenizer as GPT models!")